Mount google drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset_path = "/content/drive/MyDrive/Face_Dataset"

In [ ]:
#check the folder
import os
print(os.listdir(dataset_path))


['Elon Musk', 'Fida', 'Shundar Pichai', 'Zuhayr', 'Masrafi', 'Mofidul', 'Sakib', 'Messi', 'Alian', 'Zayan']


Import Libraries


In [3]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix


In [5]:
#Load Dataset
dataset_path = "/content/drive/MyDrive/Face_Dataset"  # update if different

X, y = [], []

for label in os.listdir(dataset_path):
    folder = os.path.join(dataset_path, label)
    if not os.path.isdir(folder):
        continue

    for file in os.listdir(folder):
        img_path = os.path.join(folder, file)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # grayscale

        if img is None:  # skip unreadable files
            continue

        img = cv2.resize(img, (100, 100))  # resize to fixed size
        X.append(img.flatten())            # flatten image to vector
        y.append(label)                    # use folder name as label

X = np.array(X)
y = np.array(y)

print("Dataset loaded:", X.shape, "labels:", len(np.unique(y)))




Dataset loaded: (178, 10000) labels: 10


In [6]:
#Split dataset
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


Train: (124, 10000) Val: (27, 10000) Test: (27, 10000)


In [7]:
#Scale+PCA
# scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# PCA
pca = PCA(0.95)  # keep 95% variance
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Reduced dimensions:", X_train_pca.shape[1])


Reduced dimensions: 75


In [8]:
#Train KNN and tune hyperparameters
best_score = 0
best_k = None

for k in range(1, 15):  # test k from 1–15
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_pca, y_train)
    score = knn.score(X_val_pca, y_val)

    print(f"k={k}, val acc={score:.3f}")

    if score > best_score:
        best_score = score
        best_k = k

print("Best k =", best_k, "with acc =", best_score)


k=1, val acc=0.815
k=2, val acc=0.815
k=3, val acc=0.815
k=4, val acc=0.815
k=5, val acc=0.815
k=6, val acc=0.778
k=7, val acc=0.741
k=8, val acc=0.704
k=9, val acc=0.778
k=10, val acc=0.741
k=11, val acc=0.704
k=12, val acc=0.741
k=13, val acc=0.741
k=14, val acc=0.741
Best k = 1 with acc = 0.8148148148148148


In [9]:
#Train final model with best k
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(np.vstack([X_train_pca, X_val_pca]), np.hstack([y_train, y_val]))  # train on train+val


KNeighborsClassifier(n_neighbors=1)

In [10]:
#Evalute on test set
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict labels for test set
y_pred = knn.predict(X_test_pca)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print("✅ Test Accuracy:", acc)

# Detailed report
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


✅ Test Accuracy: 0.7407407407407407

Classification Report:
                   precision    recall  f1-score   support

         Alian_1       1.00      1.00      1.00         3
     Elon_musk_5       1.00      0.33      0.50         3
          Fida_2       1.00      1.00      1.00         2
      Mashrafi_8       1.00      0.50      0.67         2
         Messi_3       0.67      0.67      0.67         3
       Mofidul_0       1.00      0.67      0.80         3
         Sakib_9       1.00      1.00      1.00         3
Shundar_pichai_4       0.50      0.67      0.57         3
         Zayan_7       0.60      1.00      0.75         3
        Zuhayr_6       0.33      0.50      0.40         2

        accuracy                           0.74        27
       macro avg       0.81      0.73      0.74        27
    weighted avg       0.81      0.74      0.74        27


Confusion Matrix:
 [[3 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 1 1]
 [0 0 2 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 1]
 [0 0 0 0 2 